In [ ]:
# ==========================================================
#  PREDICTION & SUBMISSION NOTEBOOK
# ==========================================================
#  OBJECTIVE:
# - Load trained model
# - Generate predictions
# - Save output for evaluation

In [22]:
# ==========================================================
#  STEP 1 — LOAD LIBRARIES
# ==========================================================
import pandas as pd
import pickle

In [23]:
# ==========================================================
#  STEP 2 — LOAD TEST DATA
# ==========================================================
test_df = pd.read_csv("../data/test.csv")

#  Keep a copy of actual values (IMPORTANT for evaluation)
#  STUDENT TASK: Change target if needed

TARGET_COLUMN = "target_churn"   #  CHANGE THIS

y_true = test_df[TARGET_COLUMN]

In [25]:
import numpy as np
# ==========================================================
#  STEP 3 — LOAD MODEL
# ==========================================================
model = pickle.load(open("../models/model.pkl", "rb"))

# ==========================================================
# STEP 4 — PREPROCESS DATA
# ==========================================================

# Fill missing values
test_df.fillna(0, inplace=True)

# Convert categorical variables
test_df = pd.get_dummies(test_df, drop_first=True)

# ==========================================================
# 🔥 STEP 4.1 — RECREATE FEATURES (VERY IMPORTANT)
# ==========================================================

# test_df["spend_per_transaction"] = test_df["total_spend"] / (test_df["num_transactions"] + 1)

# test_df["activity_score"] = test_df["num_transactions"] / (test_df["tenure_months"] + 1)

# test_df["engagement_ratio"] = test_df["last_login_days"] / (test_df["tenure_months"] + 1)

# test_df["high_spender_flag"] = (test_df["total_spend"] > test_df["total_spend"].median()).astype(int)

# test_df["frequent_user_flag"] = (test_df["num_transactions"] > test_df["num_transactions"].median()).astype(int)

# test_df["customer_value_score"] = (
#     test_df["total_spend"] * test_df["num_transactions"]
# ) / (test_df["tenure_months"] + 1)


# 1. SPEND VELOCITY - How much they spend per month on average
# Higher values indicate more valuable customers, lower values may indicate declining interest
test_df['spend_velocity'] = test_df['total_spend'] / test_df['tenure_months']
 
# 2. TRANSACTION FREQUENCY - Transactions per month
# Higher frequency = more engaged, lower = potential churn risk
test_df['transaction_frequency'] = test_df['num_transactions'] / test_df['tenure_months']
 
# 3. ENGAGEMENT SCORE - Composite metric of customer engagement
# Considers activity status, recent login, and support needs
# Lower scores indicate disengagement (churn risk)
test_df['engagement_score'] = (
    (test_df['is_active'] * 40) +  # Active status worth 40 points
    (np.maximum(30 - test_df['last_login_days'], 0)) +  # Recent login worth up to 30 points
    (np.maximum(20 - test_df['support_tickets'] * 2, 0))  # Fewer tickets = better (max 20 points)
)
 
# 4. SUPPORT INTENSITY - Support tickets per month
# High values may indicate dissatisfaction
test_df['support_intensity'] = test_df['support_tickets'] / test_df['tenure_months']
 
# 5. INACTIVITY RATIO - How long since login relative to tenure
# Higher values indicate growing disengagement
test_df['inactivity_ratio'] = test_df['last_login_days'] / test_df['tenure_months']
 
# 6. IS_HIGH_VALUE - Binary indicator for high-value customers
# Based on total spend above 75th percentile
spend_threshold = test_df['total_spend'].quantile(0.75)
test_df['is_high_value'] = (test_df['total_spend'] >= spend_threshold).astype(int)
 
# 7. TRANSACTION VALUE CONSISTENCY - Ratio of avg transaction to monthly spend
# Helps identify if customers are making consistent purchases
test_df['transaction_consistency'] = test_df['avg_transaction_value'] / test_df['monthly_spend']
 
# 8. LOYALTY INDICATOR - Combination of tenure and activity
# Long tenure + active = loyal customer
test_df['loyalty_score'] = test_df['tenure_months'] * (test_df['is_active'] + 0.5)
 
# 9. CHURN RISK SCORE - Composite risk indicator
# Higher scores = higher churn risk
test_df['churn_risk_score'] = (
    (test_df['is_active'] == 0) * 30 +  # Not active
    (test_df['last_login_days'] > 14) * 20 +  # Haven't logged in recently
    (test_df['support_tickets'] > 5) * 15 +  # High support needs
    (test_df['tenure_months'] < 6) * 25 +  # New customers at risk
    (test_df['transaction_frequency'] < 1) * 10  # Low transaction frequency
)
 
# 10. SPENDING TREND INDICATOR - Comparing monthly spend to average transaction value
# Can indicate if recent behavior differs from historical patterns
test_df['spend_pattern_ratio'] = test_df['monthly_spend'] / (test_df['avg_transaction_value'] + 1)

In [26]:
# ==========================================================
#  STEP 5 — ALIGN COLUMNS WITH TRAINING DATA
# ==========================================================
# Load training columns
train_cols = pd.read_csv("../data/processed_train.csv").drop(columns=[TARGET_COLUMN]).columns

# Align columns
test_df = test_df.reindex(columns=train_cols, fill_value=0)

In [27]:
# ==========================================================
#  STEP 6 — GENERATE PREDICTIONS
# ==========================================================
# predictions = model.predict(test_df)

probs = model.predict_proba(test_df)[:, 1]

predictions = (probs > 0.2).astype(int)

In [28]:
# just testing the threshold which one is better (usually we dont do this but for testing it helps)
for t in [0.5, 0.3, 0.2, 0.15]:
    preds = (probs > t).astype(int)
    print(f"\nThreshold {t}")
    print(pd.Series(preds).value_counts())


Threshold 0.5
0    964
1     36
Name: count, dtype: int64

Threshold 0.3
0    927
1     73
Name: count, dtype: int64

Threshold 0.2
0    877
1    123
Name: count, dtype: int64

Threshold 0.15
0    790
1    210
Name: count, dtype: int64


In [29]:
print(pd.Series(predictions).value_counts())

0    877
1    123
Name: count, dtype: int64


In [30]:
from sklearn.metrics import classification_report

predictions = (probs > 0.2).astype(int)

print(classification_report(y_true, predictions))

              precision    recall  f1-score   support

           0       0.91      0.91      0.91       882
           1       0.34      0.36      0.35       118

    accuracy                           0.84      1000
   macro avg       0.63      0.63      0.63      1000
weighted avg       0.85      0.84      0.84      1000



In [31]:
# ==========================================================
#  STEP 7 — CREATE OUTPUT FILE
# ==========================================================
#  IMPORTANT:
# This format is REQUIRED for evaluation system

submission = pd.DataFrame({
    "actual": y_true,
    "prediction": predictions
})

submission.tail(20)

,actual,prediction
980,0,1
981,1,0
982,0,0
983,0,0
984,0,0
985,0,0
986,0,0
987,0,0
988,0,0
989,0,0


In [32]:
# ==========================================================
#  SAVE FINAL PREDICTIONS (MANDATORY STEP)
# ==========================================================

# 👉 STUDENT TASK:
# Replace YOURNAME with your actual name
# Example:
# "harshit_predictions.csv"

FILE_NAME = "shadrack_itulah.csv"   # 🔥 CHANGE THIS

# 👉 Save inside outputs folder (DO NOT CHANGE PATH)
submission.to_csv(f"../outputs/{FILE_NAME}", index=False)

print("✅ File saved successfully!")

# ==========================================================
# 📌 IMPORTANT INSTRUCTIONS
# ==========================================================
# ✔ File MUST be saved in: outputs/
# ✔ File name MUST follow: YOURNAME_predictions.csv
# ✔ Columns MUST be: actual, prediction
# ✔ Do NOT change column names

# ==========================================================
# 🎯 FINAL CHECK
# ==========================================================
# Go to file explorer → outputs/
# Verify your file is present there

✅ File saved successfully!


In [33]:
from sklearn.metrics import accuracy_score, classification_report



print("Random Forest Accuracy:", accuracy_score(y_true, predictions))
print(classification_report(y_true, predictions))

Random Forest Accuracy: 0.843
              precision    recall  f1-score   support

           0       0.91      0.91      0.91       882
           1       0.34      0.36      0.35       118

    accuracy                           0.84      1000
   macro avg       0.63      0.63      0.63      1000
weighted avg       0.85      0.84      0.84      1000

